# 03 — NLP: Sentiment & Qualitative Risk

**AI Equity Research Lab** — FGV EAESP (Aula 4)

This notebook:
1. Collects news texts for each ticker (Google News RSS)
2. Runs transformer sentiment + keyword risk analysis
3. Builds a textual index (0-100) and merges it with the fundamental score
4. Visualises: sentiment heatmap, word clouds, updated ranking with deltas

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)
%matplotlib inline

print(f"Project root: {PROJECT_ROOT}")

## 1. Text Collection

In [ ]:
from src.nlp.collector import run as run_collector

all_texts = run_collector()  # uses cache if files already exist

In [ ]:
# Collection summary
tickers = ["ITUB4", "BBDC4", "BBAS3", "SANB11", "ABCB4",
           "EGIE3", "EQTL3", "CPFE3", "TAEE11", "CMIG4"]

summary_rows = []
for ticker in tickers:
    path = RAW_DIR / f"texts_{ticker}.json"
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            docs = json.load(f)
        sources = {}
        dates = []
        for d in docs:
            src = d.get("source", "unknown")
            sources[src] = sources.get(src, 0) + 1
            if d.get("date"):
                dates.append(d["date"])
        date_range = f"{min(dates)} to {max(dates)}" if dates else "N/A"
        summary_rows.append({
            "ticker": ticker,
            "total_docs": len(docs),
            "sources": str(sources),
            "date_range": date_range,
        })

summary_df = pd.DataFrame(summary_rows)
print("Text Collection Summary")
print("=" * 80)
print(summary_df.to_string(index=False))

## 2. Sentiment Analysis

In [ ]:
from src.nlp.sentiment import run as run_sentiment

sentiment = run_sentiment()

In [ ]:
sentiment = pd.read_parquet(PROCESSED_DIR / "sentiment_scores.parquet")
print(f"Shape: {sentiment.shape}")
sentiment

## 3. Sentiment Heatmap

In [ ]:
heat_cols = ["positive_score", "negative_score", "neutral_score",
             "keyword_score", "sentiment_composite"]

heat_data = sentiment.set_index("ticker")[heat_cols].copy()

# Normalise each column to 0-1 for colour mapping
heat_norm = heat_data.copy()
for col in heat_norm.columns:
    cmin, cmax = heat_norm[col].min(), heat_norm[col].max()
    if cmax != cmin:
        heat_norm[col] = (heat_norm[col] - cmin) / (cmax - cmin)
    else:
        heat_norm[col] = 0.5

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    heat_norm,
    annot=heat_data.round(3).values,  # show raw values in cells
    fmt="",
    cmap="RdYlGn",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Normalized (0=min, 1=max)"},
)
ax.set_title("Sentiment Scores — Transformer + Keywords", fontsize=14)
ax.set_ylabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 4. Word Clouds — Top Positive & Negative Keywords

In [ ]:
from wordcloud import WordCloud

# Aggregate keyword counts across all tickers
from src.nlp.sentiment import RISK_KEYWORDS, _load_texts

pos_freq = {}
neg_freq = {}

for ticker in tickers:
    texts = _load_texts(ticker)
    for doc in texts:
        text_lower = doc.get("text", "").lower()
        for kw, weight in RISK_KEYWORDS.items():
            count = text_lower.count(kw.lower())
            if count > 0:
                if weight > 0:
                    pos_freq[kw] = pos_freq.get(kw, 0) + count * weight
                else:
                    neg_freq[kw] = neg_freq.get(kw, 0) + count * abs(weight)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

if pos_freq:
    wc_pos = WordCloud(
        width=600, height=300, background_color="white",
        colormap="Greens", max_words=30,
    ).generate_from_frequencies(pos_freq)
    ax1.imshow(wc_pos, interpolation="bilinear")
ax1.set_title("Positive Keywords", fontsize=14, color="green")
ax1.axis("off")

if neg_freq:
    wc_neg = WordCloud(
        width=600, height=300, background_color="white",
        colormap="Reds", max_words=30,
    ).generate_from_frequencies(neg_freq)
    ax2.imshow(wc_neg, interpolation="bilinear")
ax2.set_title("Negative / Risk Keywords", fontsize=14, color="red")
ax2.axis("off")

plt.suptitle("Keyword Frequency — All Tickers Combined", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Textual Index & Master Score

In [ ]:
from src.nlp.textual_index import run as run_textual

master = run_textual()

In [ ]:
master = pd.read_parquet(PROCESSED_DIR / "master_scores_2024.parquet")
display_cols = ["ticker", "sector", "fundamental_score", "textual_index",
                "final_score", "rank_overall", "final_rank", "rank_delta"]
master[display_cols]

## 6. Updated Ranking — Top 5 & Bottom 5

In [ ]:
master_sorted = master.sort_values("final_rank")

# Horizontal bar chart
sector_colors = {
    "Bancos": "#2563eb",
    "Energia El\u00e9trica": "#16a34a",
}

fig, ax = plt.subplots(figsize=(12, 6))

y_pos = range(len(master_sorted))
bar_colors = [sector_colors.get(s, "#888") for s in master_sorted["sector"]]

bars = ax.barh(
    [f"#{int(r['final_rank'])} {r['ticker']}" for _, r in master_sorted.iterrows()],
    master_sorted["final_score"],
    color=bar_colors, edgecolor="white", linewidth=0.5,
)

# Annotate with score and delta
for bar, (_, row) in zip(bars, master_sorted.iterrows()):
    delta = int(row["rank_delta"])
    delta_str = f" ({'+' if delta > 0 else ''}{delta})" if delta != 0 else ""
    ax.text(
        bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
        f"{row['final_score']:.1f}{delta_str}",
        va="center", fontsize=10, fontweight="bold",
    )

ax.set_xlabel("Final Score (70% Fundamental + 30% Textual)")
ax.set_title("Master Ranking 2024 — Fundamental + NLP Sentiment", fontsize=14)
ax.set_xlim(0, 85)
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2563eb", label="Bancos"),
    Patch(facecolor="#16a34a", label="Energia El\u00e9trica"),
]
ax.legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 70)
print("TOP 5")
print("=" * 70)
for _, r in master_sorted.head(5).iterrows():
    delta = int(r["rank_delta"])
    arrow = f"({'+' if delta > 0 else ''}{delta})" if delta != 0 else "(=)"
    print(f"  #{int(r['final_rank'])} {r['ticker']:8s} "
          f"final={r['final_score']:.1f}  fund={r['fundamental_score']:.1f}  "
          f"text={r['textual_index']:.1f}  {arrow}")

print()
print("=" * 70)
print("BOTTOM 5")
print("=" * 70)
for _, r in master_sorted.tail(5).iterrows():
    delta = int(r["rank_delta"])
    arrow = f"({'+' if delta > 0 else ''}{delta})" if delta != 0 else "(=)"
    print(f"  #{int(r['final_rank'])} {r['ticker']:8s} "
          f"final={r['final_score']:.1f}  fund={r['fundamental_score']:.1f}  "
          f"text={r['textual_index']:.1f}  {arrow}")

## 7. Analysis: How the Textual Index Changed the Ranking

The textual index (30% weight) captures qualitative signals from recent news that the
purely quantitative fundamental score (70%) cannot see. Key observations:

**Winners (climbed in ranking):**
- Tickers with strong positive news sentiment and/or frequent positive keywords
  (e.g. *lucro*, *crescimento*, *dividendo*, *recorde*) saw their final score lifted.
- ITUB4, which was penalised in the fundamental score by missing net_income data,
  recovered positions thanks to consistently positive news coverage.

**Losers (dropped in ranking):**
- Tickers with weaker or mixed news sentiment relative to peers lost positions.
- The keyword analysis flagged specific risk terms (e.g. *inadimplência*, *endividamento*)
  for some tickers, pulling their textual index below the group average.

**Methodology notes:**
- The transformer model (`distilbert-base-multilingual-cased-sentiments-student`) natively
  handles Portuguese text and produces positive/negative/neutral confidence scores.
- Recency weighting (half-life = 90 days) ensures recent news matters more.
- The keyword dictionary is intentionally conservative (18 terms) — it acts as a
  domain-specific overlay, not a replacement for the learned model.
- CVM Fatos Relevantes were not obtainable via automated scraping (ASP.NET ViewState);
  Google News RSS served as the primary text source.

## 8. Data Quality Check

In [ ]:
tables = {
    "sentiment_scores": pd.read_parquet(PROCESSED_DIR / "sentiment_scores.parquet"),
    "textual_index": pd.read_parquet(PROCESSED_DIR / "textual_index.parquet"),
    "master_scores_2024": pd.read_parquet(PROCESSED_DIR / "master_scores_2024.parquet"),
}

print("=" * 60)
print("DATA QUALITY SUMMARY — NLP Module")
print("=" * 60)
for name, df in tables.items():
    print(f"\n{name}")
    print(f"  Shape: {df.shape[0]} rows x {df.shape[1]} cols")
    numeric_cols = df.select_dtypes(include="number")
    if not numeric_cols.empty:
        missing = numeric_cols.isna().sum()
        if missing.sum() == 0:
            print("  Missing numeric values: NONE")
        else:
            for col, n in missing.items():
                if n > 0:
                    print(f"  {col}: {n} NaN")